
# CLD data dictionary

checking data dictionary, and if the variable names match the vensim models

last update: 06/19/2024


In [97]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# READ I/O

In [98]:
from spellchecker import SpellChecker
from pathlib import Path

import numpy as np
import os 
import pickle
import pandas as pd
import sys
import networkx as nx
import matplotlib.pyplot as plt
import gravis as gv
import json

In [109]:

# Get the current working directory
cwd = Path.cwd()

# Get the path to the parent directory
parent_dir = cwd.parent

# Add the parent directory to the Python path
sys.path.append(str(parent_dir))

# Now you can import the data_io module from src
import src.data_io as data_io
import src.cld_nx as cld_nx
import src.parse_cld as parse_cld

from src.vensim_parser import CLDParser

# Read Data

In [101]:
print(os.getcwd())


/Users/gl665/PycharmProjects/Phrame_MentalMaps/notebook


## read raw data

In [99]:
# SYNCH 
dataset_folder_path = 'results/SYNCH/JSON_file/RC/clean'

# PHRAME
# dataset_folder_path = 'results/PHRAME/JSON_file/RC'

In [107]:
# Go one level up from the current directory
cwd = os.getcwd()

parent_dir = Path(cwd).parent
print(parent_dir)

my_data= {}
folder_path = str(parent_dir / dataset_folder_path)


cld_data = data_io.read_json(folder_path, my_data)


/Users/gl665/PycharmProjects/Phrame_MentalMaps
load: Passages Clean_edited.json
load: Wavi combined clean_edited.json


# Read Data Definition Dictionary

In [100]:
input_file = 'data/SYNCH/RC/excel_sheets/Rapid City SYNCH Models data dictionary_06_19.xlsx'

# TODO: check only Passage and Wavi First 
# Data Dictionary 
# Go one level up from the current directory
cwd = os.getcwd()


file_path = str(parent_dir / input_file)

# Read all sheets into a dictionary of DataFrames
excel_data = pd.read_excel(file_path, sheet_name=None)

# Print the keys (sheet names) of the dictionary
print("Sheet names:", list(excel_data.keys()))


# print(f"\nData from '{sheet_name}' sheet:\n{df.head()}")

Sheet names: ['full', 'Passages', 'WAVI (combined)', 'Full Circle (patient combined)', 'DSS (combined)', 'justice- drugs', 'Justice - IPV', 'full circle (provider)', 'E model 5 7 2020', 'FM 5 122020', 'nat model', 'j model 7 13 20', 'c model 7 6 20', 'reentry', 'RC Diversion', 'RC Clean', 'RC Legal', 'RC Full Circle', 'description of sessions', 'Sheet1']


In [108]:
# Access a specific sheet
sheet_name = "Passages"
df = excel_data[sheet_name]

print(f"\nData from '{sheet_name}':")
df.head()


Data from 'Passages':


,variable,changed variable name,Definition,notes
0,Accountability,NaN,Being held responsible for one's actions by ot...,NaN
1,Alcohol use,Alcohol Misuse,using alcohol - in reference to misuse,NaN
2,coping skills,NaN,Strategies and ability to carry out strategies...,NaN
3,Dire situations,NaN,Long-term Situtations/experiences that cause e...,NaN
4,Drugs,drug misuse,using drugs -in reference to misuse,NaN


# matching the excel sheet with the vensim file 

In [96]:
names_from_excel = df["changed variable name"].fillna(df["variable"]).tolist()

df['final_name'] = df["changed variable name"].fillna(df["variable"]).str.lower()


print("Numbers of variables in the excel file: ", len(names_from_excel))
print(names_from_excel)

Numbers of variables in the excel file:  16
['Accountability', 'Alcohol Misuse', 'coping skills', 'Dire situations', 'drug misuse', 'death', 'Family Emotional Health', 'losing family and children through justice or child welfare system involvement', 'belief in a higher power', 'incarceration', 'intimate partner violence', 'Mental Health', 'harmful romantic partners', 'Re-entry facility', 'Supportive People', 'Resources that provide structure and support']


In [102]:
df.head(4)

,variable,changed variable name,Definition,notes,final_name
0,Accountability,NaN,Being held responsible for one's actions by ot...,NaN,accountability
1,Alcohol use,Alcohol Misuse,using alcohol - in reference to misuse,NaN,alcohol misuse
2,coping skills,NaN,Strategies and ability to carry out strategies...,NaN,coping skills
3,Dire situations,NaN,Long-term Situtations/experiences that cause e...,NaN,dire situations


In [105]:
# dict_cld = {name_file: nxgraph}

dict_cld = {}

for k, v in my_data.items():
    cld_list, G_cld = cld_nx.get_cld_graph(my_data, k)
    # cld_name = dict_file_rename.get(k, "")
    dict_cld[k] = G_cld

for k, v in dict_cld.items(): 
    print(k, ":", v)

Passages Clean_edited.json : DiGraph with 18 nodes and 61 edges
Wavi combined clean_edited.json : DiGraph with 35 nodes and 78 edges


In [110]:
cld_var_names  = list(dict_cld['Passages Clean_edited.json'].nodes())
cld_var_names

['alcohol misuse',
 'intimate partner violence',
 'harmful romantic partners',
 'mental health',
 'alcohol exposed pregnancy',
 'family emotional health',
 'drug misuse',
 'incarceration',
 're-entry facility',
 'accountability',
 'supportive people',
 'healing',
 'belief in a higher power',
 'losing family and children through justice or child welfare system involvement',
 'resources that provide structure and support',
 'dire situations',
 'death',
 'coping skills']

# check if the data dictionary and the CLD data has the exact same name

In [106]:
# check how much variables overlapped between the two list of all variable names 
overlap = [var for var in cld_var_names if var.lower() in df['final_name'].tolist()]
print("Overlapping variables between all the variables names used in the RC models and the handcraft model: ", len(overlap))


for i, name in enumerate(overlap, start=1):
    print(f"{i}. {name}")


# check not overlapped variables 
not_overlap = [var for var in cld_var_names if var.lower() not in df['final_name'].tolist()]

print("Missing variables:", not_overlap)

Overlapping variables between all the variables names used in the RC models and the handcraft model:  16
1. alcohol misuse
2. intimate partner violence
3. harmful romantic partners
4. mental health
5. family emotional health
6. drug misuse
7. incarceration
8. re-entry facility
9. accountability
10. supportive people
11. belief in a higher power
12. losing family and children through justice or child welfare system involvement
13. resources that provide structure and support
14. dire situations
15. death
16. coping skills
Missing variables: ['alcohol exposed pregnancy', 'healing']


In [103]:
# check not overlapped variables 
not_overlap = [var for var in df['final_name'] if var.lower() not in cld_var_names]

print("Missing variables:", not_overlap)

Missing variables: []


# Create Label data

In [104]:
# get dictionary for df['final_name'] and df['definition']
# dict_name_def = dict(zip(df['final_name'], df['Definition']))
# file_name = 'data_dictionary_passages.json'

# folder_path = str(parent_dir / 'results/SYNCH/JSON_file/RC/results sheets/data_dictionary/'/file_name )

# with open(folder_path, 'w') as f:
#     json.dump(dict_name_def, f)
